# BRICK Modeling with RDFlib 

example script from brickschema github repo https://github.com/BrickSchema/Brick/blob/master/examples/example1/generate.py

In [4]:
# import
from pathlib import Path
from rdflib import RDF, Namespace, Graph, Literal, BNode, XSD

In [10]:
# Parameters
ttl_filepath = Path.cwd().joinpath('brick_eg.ttl')


In [9]:
bldg_graph = Graph()
BLDG = Namespace('urn:bldg/')
BRICK = Namespace('https://brickschema.org/schema/Brick#')
REC = Namespace('https://w3id.org/rec#')
UNIT = Namespace("http://qudt.org/vocab/unit/")
REF = Namespace("https://brickschema.org/schema/Brick/ref#")
BACNET = Namespace("http://data.ashrae.org/bacnet")

bldg_graph.bind('bldg', BLDG)
bldg_graph.bind('brick', BRICK)
bldg_graph.bind('rec', REC)
bldg_graph.bind('unit', UNIT)
bldg_graph.bind('ref', REF)
bldg_graph.bind('bacnet', BACNET)

# spaces
bldg_graph.add((BLDG['suite202_zone'], RDF.type, REC.HVACZone))
bldg_graph.add((BLDG['get_room'], RDF.type, REC.Room))

# HVAC system
bldg_graph.add((BLDG['RTU1'], RDF.type, BRICK.RTU))
bldg_graph.add((BLDG['get_flair_vent'], RDF.type, BRICK.Damper))

# sensors
bldg_graph.add((BLDG['get_room_occ'], RDF.type, BRICK.Occupancy_Sensor))
bldg_graph.add((BLDG['get_room_occ'], RDF.type, BRICK.Occupancy_Sensor))

bldg_graph.add((BLDG['get_room_tstat'], RDF.type, BRICK.Thermostat_Equipment))
bldg_graph.add((BLDG['get_room_setpoint'], RDF.type, BRICK.Air_Temperature_Setpoint))

bldg_graph.add((BLDG['get_room_temperature'], RDF.type, BRICK.Air_Temperature_Sensor))
bldg_graph.add((BLDG['get_room_temperature'], BRICK.hasUnit, UNIT.DEG_C))

bldg_graph.add((BLDG['suite202_tstat'], RDF.type, BRICK.Thermostat_Equipment))
bldg_graph.add((BLDG['suite202_setpoint'], RDF.type, BRICK.Air_Temperature_Setpoint))
bldg_graph.add((BLDG['suite202_temperature'], RDF.type, BRICK.Air_Temperature_Sensor))

# define the edges
bldg_graph.add((BLDG['RTU1'], BRICK.feeds, BLDG['suite202_zone']))
bldg_graph.add((BLDG['suite202_zone'], BRICK.hasPart, BLDG['get_room']))

bldg_graph.add((BLDG['get_room_tstat'], BRICK.hasPoint, BLDG['get_room_setpoint']))
bldg_graph.add((BLDG['get_room_tstat'], BRICK.hasPoint, BLDG['get_room_temperature']))

bldg_graph.add((BLDG['suite202_tstat'], BRICK.hasPoint, BLDG['suite202_setpoint']))
bldg_graph.add((BLDG['suite202_tstat'], BRICK.hasPoint, BLDG['suite202_temperature']))

# define a bacnet device
sample_device = BLDG["sample-device"]
bldg_graph.add((sample_device, RDF.type, BACNET.BACnetDevice))
bldg_graph.add((sample_device, BACNET["device-instance"], Literal(123)))
# create a blanknode for external reference
port_bnode = BNode()
bldg_graph.add((sample_device, BACNET.hasPort, port_bnode))
bldg_graph.add((port_bnode, RDF.type, BACNET.Port))
# Notice the dictionary syntax ["..."] is used here because of the dot in 'NetworkType.ipv4'
bldg_graph.add((port_bnode, BACNET["network-type"], BACNET["NetworkType.ipv4"]))
# Use XSD.hexBinary for the specific datatype requirements
bldg_graph.add((port_bnode, BACNET["ip-address"], Literal("C0A80164", datatype=XSD.hexBinary)))
bldg_graph.add((port_bnode, BACNET["ip-default-gateway"], Literal("C0A80101", datatype=XSD.hexBinary)))


# create a blank node as external reference
bacnet_ref = BNode()
bldg_graph.add((BLDG['get_room_temperature'], REF.hasExternalReference, bacnet_ref))
bldg_graph.add((bacnet_ref, BACNET["object-identifier"], Literal("device,999", datatype=BACNET.objectIdentifier)))
bldg_graph.add((bacnet_ref, BACNET["object-name"], Literal("BLDG-Z410-ZATS")))
bldg_graph.add((bacnet_ref, BACNET.objectOf, BLDG["sample-device"]))

# timeseries external reference
ts_ref = BNode()
bldg_graph.add((BLDG['get_room_temperature'], REF.hasExternalReference, ts_ref))
bldg_graph.add((ts_ref, REF.hasTimeseriesId, Literal("756e1623-914f-4415-9000-b1b10ce8f5c9")))
bldg_graph.add((ts_ref, REF.storedAt, Literal("postgres://1.2.3.4:5432/mydata")))

with open(ttl_filepath, 'w') as f:
    f.write(bldg_graph.serialize(format='ttl'))